# Social Indicators

## Data preparation

### Reading and normalizing data

In [94]:
import pandas as pd

In [95]:
def get_data_worldbank_xls(path):
    df = pd.read_excel(
        pd.ExcelFile(path),
        sheet_name='Data',
        skiprows=3
    )
    df.drop(columns=["Country Name", "Indicator Name", "Indicator Code", "1960", "1961", "1962", "1963", "1964", "1965", "1966", "1967", "1968", "1969", "1970", "1971", "1972", "1973", "1974", "1975", "1976", "1977", "1978", "1979", "1980", "1981", "1982", "1983", "1984", "1985", "1986", "1987", "1988", "1989", "1990", "1991", "1992", "1993", "1994", "1995", "1996", "1997", "1998", "1999", "2025"], inplace=True)
    df.dropna(
        thresh=len(df.columns) // 2,
        inplace=True
    )
    return df

def unpivot_worldbank(df):
    df = df.melt(
        id_vars=["Country Code"],
        var_name="Year",
        value_name="Value"
    )
    df.dropna(inplace=True)
    df.rename(columns={"Country Code": "Code"}, inplace=True)
    df = df.astype({"Year": int})
    return df

In [96]:
df_ed_gov_exp_pivoted = get_data_worldbank_xls('data/unprocessed/education_gov_expenditure_%.xls')
df_ed_gov_exp = unpivot_worldbank(df_ed_gov_exp_pivoted)

df_ed_post_sec_pivoted = get_data_worldbank_xls('data/unprocessed/education_post_secondary_%.xls')
df_ed_post_sec = unpivot_worldbank(df_ed_post_sec_pivoted)

df_ec_gni_pc_pivoted = get_data_worldbank_xls('data/unprocessed/economic_gni_per_capita.xls')
df_ec_gni_pc = unpivot_worldbank(df_ec_gni_pc_pivoted)

df_ec_gini_idx_pivoted = get_data_worldbank_xls('data/unprocessed/economic_gini_index.xls')
df_ec_gini_idx = unpivot_worldbank(df_ec_gini_idx_pivoted)

df_urban_pop_pivoted = get_data_worldbank_xls('data/unprocessed/urban_population_%.xls')
df_urban_pop = unpivot_worldbank(df_urban_pop_pivoted)

df_birth_rate_pivoted = get_data_worldbank_xls('data/unprocessed/birth_rate.xls')
df_birth_rate = unpivot_worldbank(df_birth_rate_pivoted)

df_birth_rate.info()

<class 'pandas.DataFrame'>
RangeIndex: 6600 entries, 0 to 6599
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    6600 non-null   str    
 1   Year    6600 non-null   int64  
 2   Value   6600 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 154.8 KB


In [97]:
df_female_labor_ratio = pd.read_csv('data/unprocessed/female_to_male_labor_force_ratio_%.csv')[['Country Code', 'Year', 'Value']]
df_female_labor_ratio.rename(columns={"Country Code": "Code"}, inplace=True)
df_female_labor_ratio = df_female_labor_ratio[ (df_female_labor_ratio['Year'] < 2025) & (df_female_labor_ratio['Year'] >= 2000)]

df_female_labor_ratio.info()

<class 'pandas.DataFrame'>
Index: 9220 entries, 1 to 13729
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    9220 non-null   str    
 1   Year    9220 non-null   int64  
 2   Value   9220 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 288.1 KB


In [98]:
df_female_parliament_sits = pd.read_csv('data/unprocessed/female_parliament_seats_%.csv')[['Country Code', 'Year', 'Value']]
df_female_parliament_sits.rename(columns={"Country Code": "Code"}, inplace=True)
df_female_parliament_sits = df_female_parliament_sits[(df_female_parliament_sits['Year'] < 2025) & (df_female_parliament_sits['Year'] >= 2000)]

df_female_parliament_sits.info()

<class 'pandas.DataFrame'>
Index: 5772 entries, 0 to 6356
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    5772 non-null   str    
 1   Year    5772 non-null   int64  
 2   Value   5772 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 180.4 KB


In [99]:
df_life_expect = pd.read_csv('data/unprocessed/life_expectancy.csv')
df_life_expect.drop(columns=["Entity"], inplace=True)
df_life_expect.rename(columns={"Life expectancy": "Value"}, inplace=True)
df_life_expect.dropna(inplace=True)
df_life_expect = df_life_expect[(df_life_expect['Year'] < 2025) & (df_life_expect['Year'] >= 2000)]

df_life_expect.info()

<class 'pandas.DataFrame'>
Index: 5904 entries, 50 to 21564
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    5904 non-null   str    
 1   Year    5904 non-null   int64  
 2   Value   5904 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 184.5 KB


In [100]:
df_rule_of_law = pd.read_excel(
    pd.ExcelFile('data/unprocessed/rule_of_law_index.xlsx'),
    sheet_name="Historical Data"
)[['Country Code', 'Year', 'WJP Rule of Law Index: Overall Score']]
df_rule_of_law.rename(columns={"Country Code": "Code", "WJP Rule of Law Index: Overall Score": "Value"}, inplace=True)
df_rule_of_law['Year'] = df_rule_of_law['Year'].astype('str')

df_rule_of_law_pivoted = df_rule_of_law.pivot_table(
    index='Code',
    columns='Year',
    values='Value'
).reset_index()
df_rule_of_law_pivoted['2013'] = df_rule_of_law_pivoted['2012-2013']
df_rule_of_law_pivoted['2018'] = df_rule_of_law_pivoted['2017-2018']
df_rule_of_law_pivoted.rename(columns={"2012-2013": "2012", "2017-2018": "2017"}, inplace=True)
df_rule_of_law_pivoted = df_rule_of_law_pivoted[['Code', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']]

df_rule_of_law = df_rule_of_law_pivoted.melt(
    id_vars='Code',
    var_name="Year",
    value_name="Value"
).dropna()
df_rule_of_law = df_rule_of_law.astype({"Year": int})
df_rule_of_law.info()

<class 'pandas.DataFrame'>
Index: 1551 entries, 2 to 1858
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    1551 non-null   str    
 1   Year    1551 non-null   int64  
 2   Value   1551 non-null   float64
dtypes: float64(1), int64(1), str(1)
memory usage: 48.5 KB


In [101]:
df_ed_gov_exp.to_csv('data/education_gov_expenditure.csv', index=False)
df_ed_post_sec.to_csv('data/education_post_secondary.csv', index=False)
df_ec_gni_pc.to_csv('data/economic_gni_per_capita.csv', index=False),
df_ec_gini_idx.to_csv('data/economic_gini_index.csv', index=False)
df_urban_pop.to_csv('data/urban_population.csv', index=False)
df_birth_rate.to_csv('data/birth_rate.csv', index=False)
df_female_labor_ratio.to_csv('data/female_labor_force_ratio.csv', index=False),
df_female_parliament_sits.to_csv('data/female_parliament_sits.csv', index=False),
df_life_expect.to_csv('data/life_expectancy.csv', index=False)
df_rule_of_law.to_csv('data/rule_of_law_score.csv', index=False)

###  Creating a single file

In [102]:
countries = pd.read_excel(
        pd.ExcelFile('data/unprocessed/birth_rate.xls'),
        sheet_name='Data',
        skiprows=3
    )[['Country Name', 'Country Code']]
countries.rename(columns={"Country Code": "Code"}, inplace=True)
countries

,Country Name,Code
0,Aruba,ABW
1,Africa Eastern and Southern,AFE
2,Afghanistan,AFG
3,Africa Western and Central,AFW
4,Angola,AGO
...,...,...
260,Kosovo,XKX
261,"Yemen, Rep.",YEM
262,South Africa,ZAF
263,Zambia,ZMB


In [103]:
df_ed_gov_exp.rename(columns={"Value": "Ed Gov Expenditure"}, inplace=True)
df_ed_post_sec.rename(columns={"Value": "Ed Post Secondary"}, inplace=True)
df_ec_gni_pc.rename(columns={"Value": "Ec GNI per capita"}, inplace=True)
df_ec_gini_idx.rename(columns={"Value": "Ec Gini Index"}, inplace=True)
df_birth_rate.rename(columns={"Value": "Birth Rate"}, inplace=True)
df_female_labor_ratio.rename(columns={"Value": "Fem Labor Ratio"}, inplace=True)
df_female_parliament_sits.rename(columns={"Value": "Fem Parliament Sits"}, inplace=True)
df_life_expect.rename(columns={"Value": "Life Expectancy"}, inplace=True)
df_rule_of_law.rename(columns={"Value": "Rule of Law"}, inplace=True)


In [104]:
def multiple_merge(dfs):
    res = dfs[0]
    for df in dfs[1:]:
        res = pd.merge(
            left=res,
            right=df,
            on=['Code', 'Year'],
            how='outer',
        )
    return res

In [108]:
df = multiple_merge([
    df_ed_gov_exp,
    df_ed_post_sec,
    df_ec_gni_pc,
    df_ec_gini_idx,
    df_birth_rate,
    df_female_labor_ratio,
    df_female_parliament_sits,
    df_life_expect,
    df_rule_of_law
])

df = pd.merge(
    left=countries,
    right=df,
    on='Code',
    how='inner'
)

df

,Country Name,Code,Year,Ed Gov Expenditure,Ed Post Secondary,Ec GNI per capita,Ec Gini Index,Birth Rate,Fem Labor Ratio,Fem Parliament Sits,Life Expectancy,Rule of Law
0,Aruba,ABW,2000,21.807711,NaN,20060.0,NaN,14.260,78.202,NaN,72.9393,NaN
1,Aruba,ABW,2001,21.323179,NaN,20340.0,NaN,13.813,NaN,NaN,73.0437,NaN
2,Aruba,ABW,2002,19.757151,NaN,19220.0,NaN,13.337,NaN,NaN,73.1355,NaN
3,Aruba,ABW,2003,NaN,NaN,21020.0,NaN,13.358,NaN,NaN,73.2362,NaN
4,Aruba,ABW,2004,14.902910,NaN,23610.0,NaN,12.540,NaN,NaN,73.2230,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
9819,Zimbabwe,ZWE,2022,NaN,NaN,2580.0,NaN,30.882,83.313,30.566,62.3601,0.391414
9820,Zimbabwe,ZWE,2023,10.688962,NaN,2550.0,NaN,30.410,83.521,30.682,62.7748,0.396527
9821,Zimbabwe,ZWE,2023,10.688962,NaN,2550.0,NaN,30.410,83.522,30.682,62.7748,0.396527
9822,Zimbabwe,ZWE,2024,16.320335,NaN,2400.0,NaN,29.891,83.535,28.090,NaN,0.396732


In [109]:
df.to_csv('data/data.csv', index=False)
